# Theta phase precession from button presses

This notebook estimates theta phase precession from the UPPT002 button-press times and the MEG epochs. It filters each MEG sensor to 2–5 Hz, extracts phase with MNE's Hilbert transform, then uses the batched circular-linear regression from circ_regress_db.py to fit a slope and correlation for each trial and sensor.

Trial results are averaged within each participant before the participant means are combined. Subject 11 is excluded. Only trials with all five valid press times and finite phase samples are included. This notebook reports descriptive participant and group means; it does not calculate per-trial or group p-values. Each trial contributes only five phase samples, so individual fits may be unstable. Use the slope for phase-drift direction: the circular-correlation coefficient's sign does not always match the slope.

import packages for loading and processing MEG epochs, circular regression, tables and figures

In [1]:
# Load the tools used for MNE epochs, array processing, result tables and plots.
import os
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from circ_regress_db import circ_regress_db_batch

define the data locations and analysis parameters

data_root can point to a local MEG export; otherwise the notebook uses the repository's network-share path. The button-press file is expected in the repository folder.

theta_freq_band sets the filter limits. Subject 11 is excluded before analysis. channel_batch_size limits memory use while channels are filtered and Hilbert-transformed. No permutation count is needed because this notebook does not calculate p-values.

In [ ]:
# Use a local data export when configured; otherwise use the shared repository location.
data_root = os.environ.get(
    "MEG_DATA_ROOT",
    r"//rdp.arc.ucl.ac.uk/ritd-ag-project-rd01pz-dbush99/Katja Button Press Data"
)

# Keep the timing file alongside this notebook.
button_press_times_file = os.path.join(
    os.getcwd(), "UPPT002_button_press_times.npz"
)

# define theta frequency band for phase extraction
theta_freq_band = [2.0, 5.0]             # theta band edges in Hz
excluded_subjects = ["11"]               # participants excluded from analysis
channel_batch_size = 8                   # channels loaded for each MNE processing batch

load and check the UPPT002 channel button-press times

The NPZ file stores uppt002_times in participant × trial × press-slot order. subject_ID records the participant row order, n_trials_by_subject gives each participant's recorded trial count, and press_slot labels presses 1 through 5. Times are in seconds relative to cue onset.

The ID helper normalizes participant labels to the repository's two-digit format if needed. The checks confirm the time reference, array dimensions and press-slot order before the participant list is made. Subject 11 is omitted from sub_ID here, so it is not included in the per-participant or group calculations.

In [ ]:
# Match IDs from the NPZ file to the two-digit labels used by the MEG notebooks.
def normalise_subject_id(id):
    return str(int(str(id).strip().lower().lstrip("s"))).zfill(2)


# Read the timing arrays and their metadata from the button-press export.
with np.load(button_press_times_file, allow_pickle=False) as press_file:
    UPPT002_times = np.asarray(press_file["uppt002_times"], dtype=float)
    subject_ID = np.asarray(
        [normalise_subject_id(id) for id in press_file["subject_ID"]]
    )
    n_trials_by_subject = np.asarray(
        press_file["n_trials_by_subject"], dtype=int
    )
    press_slot = np.asarray(press_file["press_slot"], dtype=int)
    cue_onset_after_epoch_start_s = float(
        np.asarray(press_file["cue_onset_after_epoch_start_s"]).item()
    )
    time_reference = str(press_file["time_reference"].item())

# Check that the saved times use the expected reference and array layout.
if "relative to cue onset" not in time_reference.lower():
    raise ValueError("Expected button-press times in seconds relative to cue onset")
if UPPT002_times.ndim != 3 or UPPT002_times.shape[0] != len(subject_ID):
    raise ValueError("Unexpected participant × trial × press-slot array shape")
if not np.array_equal(press_slot, np.arange(1, len(press_slot) + 1)):
    raise ValueError("Expected press slots numbered in sequence from 1")

# Keep 2-digit subject labels and a lookup to each participant's NPZ row.
n_press_slots = len(press_slot)
sub_ID = [id for id in subject_ID if id not in excluded_subjects]
subject_index = {id: index for index, id in enumerate(subject_ID)}
if not sub_ID:
    raise ValueError("No participants remain after applying excluded_subjects.")

print("Array shape:", UPPT002_times.shape)
print("Participants:", sub_ID)
print("Excluded participants:", excluded_subjects)

Array shape: (17, 240, 5)
Participants: [np.str_('05'), np.str_('06'), np.str_('08'), np.str_('09'), np.str_('10'), np.str_('13'), np.str_('14'), np.str_('15'), np.str_('16'), np.str_('17'), np.str_('18'), np.str_('20'), np.str_('21'), np.str_('24'), np.str_('25'), np.str_('28')]
Excluded participants: ['11']


load each participant's MEG epochs

filename_loader follows the repository layout: epochs are under sXX and the filename uses the participant number without its leading zero. It also checks for a local export with the FIF file at the data-root level.

The loader first opens the epochs metadata without preloading. The participant function then reads all MEG sensor samples into one array and reuses that array for its channel batches. MEG_sens contains channels whose names begin with M. mean_finite averages finite values along the requested axis and returns NaN when no values are available. This keeps file reads to one selected-channel load per participant, at the cost of holding that participant's MEG data in memory during its calculation.

In [4]:
# Locate the participant's saved epochs using the repository's subject-folder naming.
def filename_loader(id):
    subject_number = str(int(id))
    subject_folder = "s" + id
    file_name = "Subject" + subject_number + "_spm_mne-epo.fif"
    epochs_file = os.path.join(data_root, subject_folder, file_name)

    # Also allow a local export with the FIF file directly in data_root.
    if not os.path.isfile(epochs_file):
        epochs_file = os.path.join(data_root, file_name)
    if not os.path.isfile(epochs_file):
        raise FileNotFoundError("Could not find " + file_name + " under " + data_root)

    # Open metadata first; the participant function reads its MEG data once.
    epochs = mne.read_epochs(epochs_file, preload=False, verbose="ERROR")
    MEG_sens = [channel for channel in epochs.ch_names if channel.startswith("M")]
    if not MEG_sens:
        raise ValueError("No MEG sensors found in " + epochs_file)

    return epochs, MEG_sens


# Average only finite values and preserve NaN when an axis has no valid values.
def mean_finite(values, axis=0):
    values = np.asarray(values)
    counts = np.isfinite(values).sum(axis=axis)
    return np.divide(
        np.nansum(values, axis=axis),
        counts,
        out=np.full(counts.shape, np.nan, dtype=np.result_type(values.dtype, float)),
        where=counts > 0,
    )

calculate theta phase precession for one participant

The function checks that the NPZ timing rows match the saved MEG epochs, maps cue-relative press times onto the epoch time axis, and marks trials with all five valid press times.

The function reads the participant's MEG sensor data into one array before processing channel batches. Each batch is then sliced from memory and wrapped in an EpochsArray with matching sensor information; MNE filters it to theta and applies the Hilbert transform. This avoids rereading the FIF file for every channel batch. The participant's MEG array remains in memory until that participant's calculation finishes.

For each complete trial, the batched circular-linear regression fits all sensor phase series together using the slope and correlation equations from circ_regress_db.py. It only includes sensors with five finite phase samples. The function averages trial slopes and correlations within each participant and averages phase offsets circularly.

In [5]:
def participant_phase_precession(id):
    # Load this participant's epochs and find their row in the timing file.
    epochs, MEG_sens = filename_loader(id)
    data_index = subject_index[id]
    n_trials = int(n_trials_by_subject[data_index])

    # Require one timing row for every saved MEG epoch.
    if len(epochs) != n_trials:
        raise ValueError(
            "Participant " + id + " has " + str(len(epochs))
            + " MEG epochs but " + str(n_trials) + " timing rows"
        )

    # Read the required MEG sensor data once; batches below reuse this array.
    MEG_data = epochs.get_data(picks=MEG_sens)

    # Convert cue-relative press times to MNE epoch sample indices.
    press_times = UPPT002_times[data_index, :n_trials]
    cue_onset_epoch_s = epochs.times[0] + cue_onset_after_epoch_start_s
    event_times = press_times + cue_onset_epoch_s
    sample_indices = np.zeros(press_times.shape, dtype=int)
    valid_press = np.isfinite(event_times)
    sample_indices[valid_press] = np.rint(
        (event_times[valid_press] - epochs.times[0]) * epochs.info["sfreq"]
    ).astype(int)
    valid_press &= (
        (sample_indices >= 0) & (sample_indices < len(epochs.times))
    )
    # Use trials only when all five press times fall inside their MEG epoch.
    complete_trial = valid_press.sum(axis=1) == n_press_slots

    # Store one fitted slope, phase offset and correlation per trial and sensor.
    trial_slope = np.full((n_trials, len(MEG_sens)), np.nan)
    trial_phase = np.full_like(trial_slope, np.nan)
    trial_corr = np.full_like(trial_slope, np.nan)

    # Filter and transform a small channel batch at a time to limit working memory.
    for start in range(0, len(MEG_sens), channel_batch_size):
        channel_batch = MEG_sens[start:start + channel_batch_size]
        channel_indices = mne.pick_channels(
            epochs.ch_names, include=channel_batch, ordered=True
        )
        # Slice this channel batch from the participant data already in memory.
        channel_data = MEG_data[:, start:start + len(channel_batch), :]
        batch_info = mne.pick_info(epochs.info, channel_indices, copy=True)

        # Wrap the loaded data with matching sensor metadata for MNE processing.
        epochs_batch = mne.EpochsArray(
            channel_data,
            batch_info,
            events=epochs.events,
            tmin=epochs.tmin,
            baseline=None,
            verbose="ERROR",
        )
        # Filter to theta and form the complex analytic signal to extract phase.
        epochs_batch.filter(
            l_freq=theta_freq_band[0],
            h_freq=theta_freq_band[1],
            verbose="ERROR",
        ).apply_hilbert(envelope=False, verbose="ERROR")
        analytic = epochs_batch.get_data()

        # Fit all sensors in a channel batch together for each complete trial.
        for trial in np.flatnonzero(complete_trial):
            samples = sample_indices[trial]
            # Shape the selected phase values as sensors × five press slots.
            phase_by_sensor = np.angle(analytic[trial][:, samples])
            valid_sensor = np.isfinite(phase_by_sensor).all(axis=1)
            if not valid_sensor.any():
                continue

            # Vectorize the slope search across sensors; no p-values are calculated.
            slopes, phase_offsets, correlations = circ_regress_db_batch(
                press_slot, phase_by_sensor[valid_sensor]
            )
            sensor_indices = start + np.flatnonzero(valid_sensor)
            trial_slope[trial, sensor_indices] = slopes
            trial_phase[trial, sensor_indices] = phase_offsets
            trial_corr[trial, sensor_indices] = correlations

        # Release this batch's temporary arrays before processing the next channel set.
        del analytic
        del epochs_batch
        del channel_data

    # Release the full participant array before returning its sensor summaries.
    del MEG_data

    # Average trial results within this participant; wrap phase offsets circularly.
    participant_slope = mean_finite(trial_slope, axis=0)
    participant_phase = np.angle(
        mean_finite(np.exp(1j * trial_phase), axis=0)
    )
    participant_corr = mean_finite(trial_corr, axis=0)

    # Return channel geometry plus counts so group processing can verify alignment.
    picks = mne.pick_channels(epochs.ch_names, include=MEG_sens, ordered=True)
    sensor_info = mne.pick_info(epochs.info, picks, copy=True)
    summary = {
        "participant": id,
        "n_trials": n_trials,
        "n_complete_press_trials": int(complete_trial.sum()),
        "n_valid_press_times": int(valid_press.sum()),
        "n_trials_with_corr": int(np.isfinite(trial_corr).any(axis=1).sum()),
    }

    return (
        participant_slope, participant_corr, participant_phase,
        MEG_sens, sensor_info, summary
    )

process the included participants

sub_ID already excludes subject 11. Each participant is analyzed separately, and the sensor names and order are checked before values are combined. This prevents channel values from being averaged under mismatched sensor labels.

The group values are means of the participant means, so participants with more valid trials do not receive extra weight. Phase offsets are averaged as circular values.

In [6]:
# Collect each participant's sensor summaries after excluding subject 11.
participant_slope = []
participant_corr = []
participant_phase = []
participant_rows = []
group_sensors = None
group_info = None

for id in sub_ID:
    slope, corr, phase, MEG_sens, sensor_info, summary = (
        participant_phase_precession(id)
    )
    # Keep sensor order fixed so every participant contributes to the same columns.
    if group_sensors is None:
        group_sensors, group_info = MEG_sens, sensor_info
    elif MEG_sens != group_sensors:
        raise ValueError("MEG sensor order differs for participant " + id)

    # Save participant-level values and report the number of timing-complete trials.
    participant_slope.append(slope)
    participant_corr.append(corr)
    participant_phase.append(phase)
    participant_rows.append(summary)
    print(
        "Participant", id,
        "- complete trials:", summary["n_complete_press_trials"],
    )

# Rows now represent participants and columns represent MEG sensors.
participant_slope = np.asarray(participant_slope)
participant_corr = np.asarray(participant_corr)
participant_phase = np.asarray(participant_phase)
participant_summary_table = pd.DataFrame(participant_rows)

# Calculate equal-weight group means; phase offsets require a circular mean.
group_slope = mean_finite(participant_slope, axis=0)
group_corr = mean_finite(participant_corr, axis=0)
group_phase = np.angle(
    mean_finite(np.exp(1j * participant_phase), axis=0)
)

Loading data for 240 events and 14801 original time points ...
Participant 05 - complete trials: 239
Loading data for 240 events and 14801 original time points ...
Participant 06 - complete trials: 240
Loading data for 240 events and 14801 original time points ...
Participant 08 - complete trials: 240
Loading data for 240 events and 14801 original time points ...
Participant 09 - complete trials: 239
Loading data for 240 events and 14801 original time points ...
Participant 10 - complete trials: 240
Loading data for 240 events and 14801 original time points ...
Participant 13 - complete trials: 238
Loading data for 240 events and 14801 original time points ...
Participant 14 - complete trials: 240
Loading data for 240 events and 14801 original time points ...
Participant 15 - complete trials: 240
Loading data for 222 events and 14801 original time points ...
Participant 16 - complete trials: 184
Loading data for 240 events and 14801 original time points ...
Participant 17 - complete tr

summarize participant means and review the results

The tables show participant-level sensor values, participant trial counts and descriptive group means. No permutation tests, FDR corrections or p-value tables are calculated, so results should be interpreted as effect summaries rather than significance tests.

In [7]:
# Organize participant-level values and descriptive group means.
participant_corr_table = pd.DataFrame(
    participant_corr, index=sub_ID, columns=group_sensors
)
participant_slope_table = pd.DataFrame(
    participant_slope, index=sub_ID, columns=group_sensors
)

group_results = pd.DataFrame({
    "sensor": group_sensors,
    "mean_corr": group_corr,
    "mean_slope_rad_per_press": group_slope,
    "mean_phase_offset_rad": group_phase,
})

# Display trial counts, participant maps and group mean results.
display(participant_summary_table.round(4))
display(participant_corr_table.round(3))
display(participant_slope_table.round(3))
display(group_results.round(4))

,participant,n_trials,n_complete_press_trials,n_valid_press_times,n_trials_with_corr
0,05,240,239,1199,239
1,06,240,240,1200,240
2,08,240,240,1200,240
3,09,240,239,1199,239
4,10,240,240,1200,240
5,13,240,238,1198,238
6,14,240,240,1200,240
7,15,240,240,1200,240
8,16,222,184,1034,184
9,17,240,240,1200,240


,MLC11,MLC12,MLC13,MLC14,MLC15,MLC16,MLC17,MLC21,MLC22,MLC23,...,MZC02,MZC03,MZC04,MZF01,MZF02,MZF03,MZO01,MZO02,MZO03,MZP01
05,-0.007,0.034,-0.076,-0.005,-0.030,-0.030,0.045,-0.025,-0.031,-0.009,...,-0.010,0.069,0.004,0.019,-0.041,0.015,0.040,0.001,-0.014,0.045
06,0.105,0.033,-0.006,0.006,0.028,0.006,0.001,-0.019,-0.023,0.051,...,-0.058,-0.033,0.069,0.015,-0.002,-0.005,-0.060,-0.039,-0.003,-0.032
08,0.023,-0.031,0.026,-0.023,0.055,-0.074,0.009,-0.054,0.002,-0.018,...,0.011,-0.079,-0.019,0.007,-0.003,0.033,0.110,-0.018,0.068,0.089
09,-0.002,-0.027,-0.059,0.002,-0.047,0.042,0.013,-0.009,-0.035,0.048,...,-0.015,0.045,-0.017,0.006,-0.023,0.005,-0.042,0.011,-0.023,0.002
10,0.035,-0.002,-0.068,0.019,0.001,0.045,-0.074,0.014,-0.012,-0.017,...,0.011,0.016,-0.064,-0.032,-0.014,-0.034,0.014,0.071,-0.010,-0.027
13,-0.003,0.012,0.047,0.042,0.039,-0.035,-0.039,-0.038,-0.012,-0.002,...,-0.018,0.001,0.077,-0.006,-0.022,0.068,-0.011,0.002,0.069,0.001
14,-0.026,-0.037,-0.011,0.016,-0.109,0.034,-0.042,0.022,0.056,0.058,...,-0.049,-0.025,-0.085,0.022,-0.087,-0.031,-0.036,-0.021,0.004,0.012
15,-0.012,0.054,0.038,-0.033,-0.056,-0.076,0.046,-0.008,-0.005,-0.036,...,-0.018,-0.014,-0.041,-0.017,0.026,-0.037,0.005,0.056,-0.025,0.059
16,0.017,0.024,-0.055,0.020,-0.107,-0.041,0.048,-0.056,0.058,-0.002,...,0.026,-0.022,0.014,0.007,-0.022,-0.029,-0.038,0.027,-0.054,-0.016
17,0.051,0.040,-0.045,-0.002,0.046,-0.047,0.012,0.024,-0.007,-0.020,...,0.076,-0.044,0.024,0.033,-0.011,0.015,-0.005,-0.010,0.058,0.066


,MLC11,MLC12,MLC13,MLC14,MLC15,MLC16,MLC17,MLC21,MLC22,MLC23,...,MZC02,MZC03,MZC04,MZF01,MZF02,MZF03,MZO01,MZO02,MZO03,MZP01
05,0.307,0.012,-0.250,-0.258,0.114,-0.090,-0.116,-0.000,-0.309,0.001,...,-0.199,0.320,0.114,0.239,0.379,0.521,0.351,0.066,0.023,-0.025
06,0.534,0.077,-0.140,-0.295,0.410,-0.065,-0.161,-0.055,-0.136,0.355,...,0.087,0.030,0.138,0.080,0.124,0.071,-0.504,-0.081,-0.087,-0.316
08,0.013,-0.393,0.372,0.097,0.213,0.027,0.447,-0.241,-0.077,0.174,...,0.133,0.504,-0.094,-0.172,-0.070,-0.205,0.504,0.132,0.184,0.397
09,0.051,-0.115,0.048,0.013,-0.143,0.202,0.149,0.094,-0.161,0.228,...,0.061,0.363,0.196,0.192,-0.426,0.196,-0.259,0.199,-0.241,-0.112
10,0.041,-0.186,-0.063,0.042,-0.029,0.413,-0.085,0.108,0.018,-0.198,...,0.319,0.385,-0.188,-0.030,-0.032,-0.258,0.007,0.346,-0.016,-0.100
13,0.065,0.292,0.280,0.049,0.148,-0.490,-0.252,-0.339,0.090,0.025,...,-1.123,-1.424,-0.473,0.319,0.432,0.436,-0.080,-0.269,0.352,-0.256
14,-0.110,0.173,-0.186,0.046,-0.168,0.196,-0.003,0.467,0.234,0.329,...,0.113,0.467,-0.090,0.352,-0.163,-0.077,0.040,0.097,0.065,0.234
15,-0.506,0.083,-0.138,-0.071,0.212,-0.201,0.030,-0.099,-0.188,-0.223,...,-0.034,0.004,0.197,-0.205,-0.100,-0.369,0.127,0.333,0.339,0.332
16,0.235,0.341,-0.093,0.134,-0.450,0.150,0.517,-0.091,0.420,0.309,...,0.175,-0.018,0.234,0.261,0.015,-0.414,0.078,0.220,0.207,-0.094
17,0.087,0.154,-0.244,-0.084,0.205,-0.110,0.128,0.198,0.122,-0.027,...,0.219,0.241,0.457,0.253,-0.231,0.219,0.460,0.332,0.284,0.991


,sensor,mean_corr,mean_slope_rad_per_press,mean_phase_offset_rad
0,MLC11,0.0182,0.1282,-2.1288
1,MLC12,0.0099,0.0939,-2.1704
2,MLC13,-0.0124,0.0071,-3.0779
3,MLC14,0.0021,0.0809,2.8732
4,MLC15,-0.0117,0.1670,2.1257
...,...,...,...,...
268,MZF03,-0.0004,0.0696,-2.5011
269,MZO01,0.0040,0.0924,-0.6584
270,MZO02,0.0033,0.1323,-0.7685
271,MZO03,-0.0065,0.0277,-2.9193


plot the group mean for each MEG sensor type

The correlation and slope maps show the equal-weight mean across participant means. 
The displayed color limits span the full coefficient/slope range.

In [ ]:
# Use the MNE sensor metadata to split maps by magnetometer/gradiometer type.
sensor_types = [
    mne.channel_type(group_info, index)
    for index in range(len(group_sensors))
]

# Plot the group mean correlation and slope for each sensor type.
for values, label, limits in [
    (group_corr, "Mean signed circular correlation", (-1.0, 1.0)),
    (group_slope, "Mean slope (radians per press)", (-2 * np.pi, 2 * np.pi)),
]:
    for sensor_type in sorted(set(sensor_types)):
        picks = np.array([
            index for index, kind in enumerate(sensor_types)
            if kind == sensor_type
        ])
        sensor_info = mne.pick_info(group_info, picks, copy=True)
        fig, ax = plt.subplots(figsize=(7, 6))
        image, _ = mne.viz.plot_topomap(
            values[picks],
            sensor_info,
            ch_type=sensor_type,
            axes=ax,
            cmap="RdBu_r",
            contours=0,
            show=False,
        )
        # Apply one fixed colour range per measure for comparable sensor maps.
        image.set_clim(*limits)
        ax.set_title(label + " (" + sensor_type + ")")
        fig.colorbar(image, ax=ax)
        plt.tight_layout()
        plt.show()